# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = "/content/drive/MyDrive/257_project"
MODEL_DIR = f"{PROJ}/smolvla_libero"
os.makedirs(f"{PROJ}/results", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Install lerobot[smolvla]

In [ ]:
!pip install -q "lerobot[smolvla,dataset]"

In [ ]:
import lerobot, torch, sys
print("python", sys.version.split()[0], "| lerobot", lerobot.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("model on Drive:", os.path.exists(MODEL_DIR))

python 3.13.16 | lerobot 0.7.0 | torch 2.14.1+cu130
GPU: Tesla T4
model on Drive: True


In [ ]:
# --- Phase 1 offline test: true vs blank vs swapped instruction ---
import random, torch, numpy as np, pandas as pd
import av
from lerobot.utils import import_utils
import_utils._require_package_cache["av"] = True
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

SMOKE = False                     # True on CPU; False on the T4 for the full run
N_EPISODES = 10                   # episodes per suite
N_FRAMES = 8 if SMOKE else 200    # frames per suite
BATCH = 4 if SMOKE else 16
REPOS = {"goal": "lerobot/libero_goal_image", "object": "lerobot/libero_object_image",
         "spatial": "lerobot/libero_spatial_image", "long": "lerobot/libero_10_image"}
device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(0); torch.manual_seed(0)

# model + processors
policy = SmolVLAPolicy.from_pretrained(MODEL_DIR).to(device).eval()
pre, post = make_pre_post_processors(
    policy.config, pretrained_path=MODEL_DIR,
    preprocessor_overrides={"device_processor": {"device": device}},
)
chunk = policy.config.chunk_size

datasets_, swap, chosen = {}, {}, {}
for suite, repo in REPOS.items():
    root = f"/content/data/{repo.replace('/', '__')}"
    m = LeRobotDatasetMetadata(repo, root=root)
    tasks = m.tasks.sort_values("task_index").index.tolist()
    swap.update({t: tasks[(i + 1) % len(tasks)] for i, t in enumerate(tasks)})   # next task in same suite
    chosen[suite] = sorted(random.sample(range(m.total_episodes), N_EPISODES))
    ds = LeRobotDataset(repo, root=root, episodes=chosen[suite],
                        delta_timestamps={"action": [i / m.fps for i in range(chunk)]})
    # sanity check: rows belong to the episodes we asked for
    got = set(int(e) for e in ds.hf_dataset["episode_index"])
    assert got == set(chosen[suite]), f"{suite}: asked {chosen[suite]}, got {sorted(got)}"
    datasets_[suite] = ds
    print(f"{suite}: {len(tasks)} tasks | {len(ds)} frames from episodes {chosen[suite]}")

def to_policy_keys(item):
    item = dict(item)
    item["observation.images.image2"] = item.pop("observation.images.wrist_image")
    return item

# sanity check: shapes match what SmolVLA expects
x = to_policy_keys(datasets_["goal"][0])
for k in ["observation.images.image", "observation.images.image2", "observation.state", "action"]:
    print(f"  {k}: {tuple(x[k].shape)}")
print("  task:", x["task"])

# run the three conditions
rows = []
for suite, ds in datasets_.items():
    idx = random.sample(range(len(ds)), N_FRAMES)
    for b in range(0, len(idx), BATCH):
        part = idx[b:b + BATCH]
        batch = torch.utils.data.default_collate([to_policy_keys(ds[i]) for i in part])
        true_tasks = list(batch["task"])
        conds = {"true": true_tasks, "blank": [""] * len(part), "swap": [swap[t] for t in true_tasks]}
        noise = torch.randn(len(part), chunk, policy.config.max_action_dim, device=device)
        time = torch.rand(len(part), device=device) * 0.998 + 0.001
        out = {}
        with torch.no_grad():
            for c, tasks in conds.items():
                bb = dict(batch); bb["task"] = tasks
                loss, _ = policy.forward(pre(bb), noise=noise, time=time, reduction="none")
                out[c] = loss.float().cpu().tolist()
        for k, i in enumerate(part):
            rows.append({"suite": suite, "frame": i, "task": true_tasks[k],
                         **{f"loss_{c}": out[c][k] for c in conds}})
        print(f"{suite}: {min(b + BATCH, len(idx))}/{len(idx)} frames done")

df = pd.DataFrame(rows)
df.to_csv(f"{PROJ}/results/phase1_offline{'_smoke' if SMOKE else ''}.csv", index=False)

# summary: mean losses and gaps
summary = df.groupby("suite")[["loss_true", "loss_blank", "loss_swap"]].mean()
summary["gap_swap"] = summary.loss_swap - summary.loss_true
summary["gap_blank"] = summary.loss_blank - summary.loss_true
print(summary.round(4))

# 95% confidence intervals for the gaps (bootstrap over frames)
rng = np.random.default_rng(0)
def ci(x, n=2000):
    means = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])
print("\n95% CI for gaps:")
for suite, g in df.groupby("suite"):
    s = ci((g.loss_swap - g.loss_true).values); b = ci((g.loss_blank - g.loss_true).values)
    print(f"  {suite:8s} gap_swap [{s[0]:.3f}, {s[1]:.3f}] | gap_blank [{b[0]:.3f}, {b[1]:.3f}]")

Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.03GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

... (25 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

long: 10 tasks | 2905 frames from episodes [37, 51, 161, 169, 181, 222, 241, 286, 350, 373]
  observation.images.image: (3, 256, 256)
  observation.images.image2: (3, 256, 256)
  observation.state: (8,)
  action: (50, 7)
  task: open the middle drawer of the cabinet
goal: 16/200 frames done
goal: 32/200 frames done
goal: 48/200 frames done
goal: 64/200 frames done
goal: 80/200 frames done
goal: 96/200 frames done
goal: 112/200 frames done
goal: 128/200 frames done
goal: 144/200 frames done
goal: 160/200 frames done
goal: 176/200 frames done
goal: 192/200 frames done
goal: 200/200 frames done
object: 16/200 frames done
object: 32/200 frames done
object: 48/200 frames done
object: 64/200 frames done
object: 80/200 frames done
object: 96/200 frames done
object: 112/200 frames done
object: 128/200 frames done
object: 144/200 frames done
object: 160/200 frames done
object: 176/200 frames done
object: 192/200 frames done
object: 200/200 frames done
spatial: 16/200 frames done
spatial: 32/200